In [ ]:
import torch
from transformers import CLIPModel, CLIPTokenizer
import open_clip
from utils_dir.backbones_utils import load_backbone_and_tokenizer, load_backbone
from collections import OrderedDict
from CoOp.clip import clip

In [3]:
import sys
import os
sys.path.append(os.path.abspath("Long-CLIP/model"))
import longclip

ImportError: attempted relative import with no known parent package

In [ ]:
test = "just want to test this out"

### Look at state dict

In [2]:
state_dict = torch.load('/home/gridsan/manderson/ovdsat/weights/RS5M_ViT-L-14.pt', map_location="cpu")

In [3]:
print(state_dict)

OrderedDict([('positional_embedding', tensor([[ 0.0015,  0.0020,  0.0004,  ..., -0.0010,  0.0007,  0.0014],
        [ 0.0044,  0.0032, -0.0005,  ...,  0.0003,  0.0015, -0.0012],
        [ 0.0017,  0.0009, -0.0005,  ..., -0.0026, -0.0009,  0.0022],
        ...,
        [ 0.0206,  0.0049, -0.0090,  ..., -0.0062, -0.0023,  0.0036],
        [ 0.0179,  0.0066, -0.0080,  ..., -0.0026, -0.0008,  0.0055],
        [ 0.0318,  0.0267,  0.0274,  ...,  0.0153,  0.0106, -0.0290]])), ('text_projection', tensor([[-0.0105,  0.0092, -0.0033,  ..., -0.0007,  0.0109, -0.0037],
        [-0.0053, -0.0047,  0.0054,  ...,  0.0230,  0.0162, -0.0063],
        [ 0.0027,  0.0098, -0.0153,  ...,  0.0068, -0.0114, -0.0091],
        ...,
        [-0.0105,  0.0009,  0.0018,  ..., -0.0167, -0.0089, -0.0136],
        [ 0.0082,  0.0086,  0.0192,  ..., -0.0101, -0.0182,  0.0039],
        [-0.0158, -0.0016, -0.0147,  ...,  0.0093,  0.0358,  0.0004]])), ('logit_scale', tensor(4.6072)), ('visual.class_embedding', tensor([ 0

In [5]:
vit = "visual.proj" in state_dict

if vit:
    vision_width = state_dict["visual.conv1.weight"].shape[0]
    vision_layers = len([k for k in state_dict.keys() if k.startswith("visual.") and k.endswith(".attn.in_proj_weight")])
    vision_patch_size = state_dict["visual.conv1.weight"].shape[-1]
    grid_size = round((state_dict["visual.positional_embedding"].shape[0] - 1) ** 0.5)
    image_resolution = vision_patch_size * grid_size
else:
    counts: list = [len(set(k.split(".")[2] for k in state_dict if k.startswith(f"visual.layer{b}"))) for b in [1, 2, 3, 4]]
    vision_layers = tuple(counts)
    vision_width = state_dict["visual.layer1.0.conv1.weight"].shape[0]
    output_width = round((state_dict["visual.attnpool.positional_embedding"].shape[0] - 1) ** 0.5)
    vision_patch_size = None
    assert output_width ** 2 + 1 == state_dict["visual.attnpool.positional_embedding"].shape[0]
    image_resolution = output_width * 32

embed_dim = state_dict["text_projection"].shape[1]
context_length = state_dict["positional_embedding"].shape[0]
vocab_size = state_dict["token_embedding.weight"].shape[0]
transformer_width = state_dict["ln_final.weight"].shape[0]
transformer_heads = transformer_width // 64
transformer_layers = len(set(k.split(".")[2] for k in state_dict if k.startswith(f"transformer.resblocks")))

In [7]:
print(vision_layers)
print(transformer_layers)

24
12


### Source CLIP

In [6]:
backbone_name = "ViT-L/14"
url = clip._MODELS[backbone_name]
model_path = clip._download(url)

In [7]:
model = torch.jit.load(model_path, map_location="cpu").eval()

In [8]:
model.visual

RecursiveScriptModule(
  original_name=VisualTransformer
  (conv1): RecursiveScriptModule(original_name=Conv2d)
  (ln_pre): RecursiveScriptModule(original_name=LayerNorm)
  (transformer): RecursiveScriptModule(
    original_name=Transformer
    (resblocks): RecursiveScriptModule(
      original_name=Sequential
      (0): RecursiveScriptModule(
        original_name=ResidualAttentionBlock
        (attn): RecursiveScriptModule(
          original_name=MultiheadAttention
          (out_proj): RecursiveScriptModule(original_name=_LinearWithBias)
        )
        (ln_1): RecursiveScriptModule(original_name=LayerNorm)
        (mlp): RecursiveScriptModule(
          original_name=Sequential
          (c_fc): RecursiveScriptModule(original_name=Linear)
          (gelu): RecursiveScriptModule(original_name=QuickGELU)
          (c_proj): RecursiveScriptModule(original_name=Linear)
        )
        (ln_2): RecursiveScriptModule(original_name=LayerNorm)
      )
      (1): RecursiveScriptModule(


In [9]:
model.transformer

RecursiveScriptModule(
  original_name=Transformer
  (resblocks): RecursiveScriptModule(
    original_name=Sequential
    (0): RecursiveScriptModule(
      original_name=ResidualAttentionBlock
      (attn): RecursiveScriptModule(
        original_name=MultiheadAttention
        (out_proj): RecursiveScriptModule(original_name=_LinearWithBias)
      )
      (ln_1): RecursiveScriptModule(original_name=LayerNorm)
      (mlp): RecursiveScriptModule(
        original_name=Sequential
        (c_fc): RecursiveScriptModule(original_name=Linear)
        (gelu): RecursiveScriptModule(original_name=QuickGELU)
        (c_proj): RecursiveScriptModule(original_name=Linear)
      )
      (ln_2): RecursiveScriptModule(original_name=LayerNorm)
    )
    (1): RecursiveScriptModule(
      original_name=ResidualAttentionBlock
      (attn): RecursiveScriptModule(
        original_name=MultiheadAttention
        (out_proj): RecursiveScriptModule(original_name=_LinearWithBias)
      )
      (ln_1): RecursiveS

In [11]:
state_dict = torch.load('/home/gridsan/manderson/ovdsat/weights/vlm4rs/openclip-fmow-4.pt', map_location="cpu")

In [13]:
model = clip.build_model(state_dict)

In [15]:
model.dtype

torch.float16

In [14]:
model

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
    (ln_pre): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): Sequential(
        (0): ResidualAttentionBlock(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=1024, out_features=1024, bias=True)
          )
          (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=1024, out_features=4096, bias=True)
            (gelu): QuickGELU()
            (c_proj): Linear(in_features=4096, out_features=1024, bias=True)
          )
          (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        )
        (1): ResidualAttentionBlock(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=1024, out_features=1024, bias=True)


### Transformers CLIP

In [2]:
clip_model, clip_tokenizer = load_backbone_and_tokenizer('clip-14')

LOADING CLIP_14!


/home/gridsan/manderson/.conda/envs/train-clip/lib/python3.8/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [3]:
clip_model

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 768)
      (position_embedding): Embedding(77, 768)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPSdpaAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e

In [4]:
clip_model.logit_scale

Parameter containing:
tensor(4.6052)

In [11]:
clip_model.text_model.final_layer_norm.weight.shape[0]

768

In [32]:
clip_model.text_model.embeddings

CLIPTextEmbeddings(
  (token_embedding): Embedding(49408, 768)
  (position_embedding): Embedding(77, 768)
)

In [24]:
clip_model.config

CLIPConfig {
  "_name_or_path": "weights/clip-vit-large-patch14",
  "architectures": [
    "CLIPModel"
  ],
  "initializer_factor": 1.0,
  "logit_scale_init_value": 2.6592,
  "model_type": "clip",
  "projection_dim": 768,
  "text_config": {
    "dropout": 0.0,
    "hidden_size": 768,
    "intermediate_size": 3072,
    "model_type": "clip_text_model",
    "num_attention_heads": 12,
    "projection_dim": 768
  },
  "torch_dtype": "float32",
  "transformers_version": "4.44.2",
  "vision_config": {
    "dropout": 0.0,
    "hidden_size": 1024,
    "intermediate_size": 4096,
    "model_type": "clip_vision_model",
    "num_attention_heads": 16,
    "num_hidden_layers": 24,
    "patch_size": 14,
    "projection_dim": 768
  }
}

In [10]:
clip_model.text_model.embeddings.position_embedding

Embedding(77, 768)

In [25]:
clip_tokens = clip_tokenizer(test, return_tensors="pt", padding=True, truncation=True)
clip_tokens = clip_tokens['input_ids']
clip_tokens

tensor([[49406,   761,  1280,   531,  1628,   589,   620, 49407]])

In [27]:
clip_text_feat = clip_model.get_text_features(clip_tokens)

In [30]:
clip_text_feat[0,:100]

tensor([ 0.4301,  0.1182,  0.0732, -0.3695,  0.3121, -0.4002,  0.2972,  0.1350,
        -0.2961,  0.5749, -0.2769,  0.0341,  0.1595,  0.0755, -0.4702,  0.2296,
         0.1203,  0.1923,  0.2482, -0.1884, -0.3011, -0.0691,  0.7934, -0.1838,
        -0.1412, -0.4564, -0.0196, -0.0365, -0.1638,  0.0470, -0.3264, -0.1455,
        -0.0970,  0.3988, -0.1224,  0.2522,  0.0453, -0.0384, -0.0026, -0.3132,
        -0.1930, -0.4049, -0.0675, -0.1064,  0.0201,  0.0335,  0.0910,  0.3549,
         0.0717, -0.4062,  0.4175, -0.1377,  0.2744,  0.1512, -0.2307, -0.1655,
         0.4032,  0.2696,  0.1790, -0.4718, -0.2223, -0.0937,  0.0310,  0.2865,
        -0.1024, -0.2110,  0.2544, -0.3426,  0.0085, -0.2316, -0.1836,  0.0764,
         0.3922, -0.0133, -0.0822,  0.1876,  0.4508,  0.1609,  0.1927, -0.2605,
        -0.0250,  0.2767, -0.2077,  0.4781,  0.4827, -0.0059, -0.3805,  0.2559,
        -0.6207, -0.1623, -0.0978, -0.3897, -0.1906,  0.1601, -0.4234,  0.3014,
         0.1316, -0.4514, -0.4234, -0.24

In [30]:
clip_tokenizer(test)['input_ids']

[49406, 761, 1280, 531, 1628, 589, 620, 49407]

### Openclip

In [7]:
openclip_model, openclip_tokenizer = load_backbone_and_tokenizer('openclip-14')

FileNotFoundError: Failed to download file (open_clip_pytorch_model.bin) for timm/vit_large_patch14_clip_224.openai. Last error: An error happened while trying to locate the file on the Hub and we cannot find the requested files in the local cache. Please check your connection and try again or make sure your Internet connection is on.

In [4]:
openclip_model, _, _ = open_clip.create_model_and_transforms('ViT-L-14', pretrained='/home/gridsan/manderson/open_clip/vit_large_patch14_clip_224.openai/open_clip_pytorch_model.bin')

/home/gridsan/manderson/.conda/envs/dassl/lib/python3.8/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [10]:
openclip_model

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-23): 24 x ResidualAttentionBlock(
          (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=1024, out_features=1024, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=1024, out_features=4096, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=4096, out_features=1024, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((1024,), eps=1e-05, elementwi

In [12]:
openclip_model.config

AttributeError: 'CLIP' object has no attribute 'config'

In [11]:
openclip_model.token_embedding

Embedding(49408, 768)

In [16]:
next(openclip_model.parameters()).dtype

torch.float32

In [17]:
openclip_model.visual.image_size

(224, 224)

In [20]:
openclip_model.visual

VisionTransformer(
  (conv1): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
  (patch_dropout): Identity()
  (ln_pre): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  (transformer): Transformer(
    (resblocks): ModuleList(
      (0-23): 24 x ResidualAttentionBlock(
        (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=1024, out_features=1024, bias=True)
        )
        (ls_1): Identity()
        (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (mlp): Sequential(
          (c_fc): Linear(in_features=1024, out_features=4096, bias=True)
          (gelu): QuickGELU()
          (c_proj): Linear(in_features=4096, out_features=1024, bias=True)
        )
        (ls_2): Identity()
      )
    )
  )
  (ln_post): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
)

In [19]:
openclip_tokens = openclip_tokenizer(test)
openclip_tokens

tensor([[49406,   761,  1280,   531,  1628,   589,   620, 49407,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0]])

In [31]:
openclip_text_feat = openclip_model.encode_text(openclip_tokens)

In [32]:
openclip_text_feat[0,:100]

tensor([ 0.4301,  0.1182,  0.0732, -0.3695,  0.3121, -0.4002,  0.2972,  0.1350,
        -0.2961,  0.5749, -0.2769,  0.0341,  0.1595,  0.0755, -0.4702,  0.2296,
         0.1203,  0.1923,  0.2482, -0.1884, -0.3011, -0.0691,  0.7934, -0.1838,
        -0.1412, -0.4564, -0.0196, -0.0365, -0.1638,  0.0470, -0.3264, -0.1455,
        -0.0970,  0.3988, -0.1224,  0.2522,  0.0453, -0.0384, -0.0026, -0.3132,
        -0.1930, -0.4049, -0.0675, -0.1064,  0.0201,  0.0335,  0.0910,  0.3549,
         0.0717, -0.4062,  0.4175, -0.1377,  0.2744,  0.1512, -0.2307, -0.1655,
         0.4032,  0.2696,  0.1790, -0.4718, -0.2223, -0.0937,  0.0310,  0.2865,
        -0.1024, -0.2110,  0.2544, -0.3426,  0.0085, -0.2316, -0.1836,  0.0764,
         0.3922, -0.0133, -0.0822,  0.1876,  0.4508,  0.1609,  0.1927, -0.2605,
        -0.0250,  0.2767, -0.2077,  0.4781,  0.4827, -0.0059, -0.3805,  0.2559,
        -0.6207, -0.1623, -0.0978, -0.3897, -0.1906,  0.1601, -0.4234,  0.3014,
         0.1316, -0.4514, -0.4234, -0.24

In [2]:
PATH_CKPT_OPENCLIP14_GPTe_1024_EPOCH50 = '/home/gridsan/manderson/open_clip/logs/2025_01_17-10_41_35-model_ViT-L-14-lr_5e-06-b_90-j_1-p_amp/checkpoints/epoch_32.pt'

In [9]:
model, _, _ = open_clip.create_model_and_transforms('ViT-L-14')
ckpt = torch.load(PATH_CKPT_OPENCLIP14_GPTe_1024_EPOCH50, map_location="cpu")

# Remove "module." prefix if it exists
new_state_dict = OrderedDict()
for k, v in ckpt['state_dict'].items():
    new_key = k.replace("module.", "")  # Remove "module." prefix
    new_state_dict[new_key] = v

# Load modified state_dict
model.load_state_dict(new_state_dict, strict=False)
model = model.visual
model.output_tokens = True
print(f'Using checkpoint {PATH_CKPT_OPENCLIP14_GPTe_1024_EPOCH50}')

Using checkpoint /home/gridsan/manderson/open_clip/logs/2025_01_17-10_41_35-model_ViT-L-14-lr_5e-06-b_90-j_1-p_amp/checkpoints/epoch_32.pt


In [5]:
print(ckpt['state_dict'])

OrderedDict([('module.positional_embedding', tensor([[ 0.0010,  0.0018,  0.0006,  ..., -0.0013,  0.0009,  0.0014],
        [ 0.0040,  0.0020, -0.0003,  ...,  0.0007,  0.0015, -0.0004],
        [ 0.0020,  0.0010, -0.0008,  ..., -0.0026, -0.0014,  0.0022],
        ...,
        [ 0.0215,  0.0056, -0.0100,  ..., -0.0061, -0.0026,  0.0036],
        [ 0.0189,  0.0066, -0.0085,  ..., -0.0027, -0.0007,  0.0055],
        [ 0.0327,  0.0282,  0.0281,  ...,  0.0157,  0.0094, -0.0306]])), ('module.text_projection', tensor([[-0.0112,  0.0093, -0.0038,  ..., -0.0016,  0.0116, -0.0036],
        [-0.0050, -0.0048,  0.0063,  ...,  0.0240,  0.0167, -0.0071],
        [ 0.0031,  0.0099, -0.0158,  ...,  0.0072, -0.0117, -0.0092],
        ...,
        [-0.0109,  0.0010,  0.0021,  ..., -0.0172, -0.0094, -0.0145],
        [ 0.0080,  0.0089,  0.0193,  ..., -0.0103, -0.0193,  0.0041],
        [-0.0160, -0.0018, -0.0150,  ...,  0.0090,  0.0380,  0.0009]])), ('module.logit_scale', tensor(4.6052)), ('module.visual.

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [17]:
# Openclip
model, tokenizer = load_backbone_and_tokenizer('openclip-14')
model = model.to(device)
prompts = 'test'
tokenized_prompts = tokenizer(prompts).to(device)
text_features = model.encode_text(tokenized_prompts).to(device)
print(text_features.shape)
#text_features = model.text_projection(text_features).to(device)

torch.Size([1, 768])


TypeError: 'Parameter' object is not callable

In [33]:
model = load_backbone('openclip-14').to(device)

In [34]:
print(model)

VisionTransformer(
  (conv1): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
  (patch_dropout): Identity()
  (ln_pre): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  (transformer): Transformer(
    (resblocks): ModuleList(
      (0-23): 24 x ResidualAttentionBlock(
        (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=1024, out_features=1024, bias=True)
        )
        (ls_1): Identity()
        (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (mlp): Sequential(
          (c_fc): Linear(in_features=1024, out_features=4096, bias=True)
          (gelu): QuickGELU()
          (c_proj): Linear(in_features=4096, out_features=1024, bias=True)
        )
        (ls_2): Identity()
      )
    )
  )
  (ln_post): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
)


In [37]:
output = model(random_image)

In [41]:
print(output[0].shape)
print(output[1].shape)

torch.Size([1, 768])
torch.Size([1, 256, 1024])


In [42]:
output[0].unsqueeze(1).shape

torch.Size([1, 1, 768])

In [21]:
print(model)

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-23): 24 x ResidualAttentionBlock(
          (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=1024, out_features=1024, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=1024, out_features=4096, bias=True)
            (gelu): QuickGELU()
            (c_proj): Linear(in_features=4096, out_features=1024, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((1024,), eps=1e-05, elementwise_affine=Tru

In [22]:
random_image = torch.rand(1, 3, 224, 224).to(device)
model.visual(random_image).shape

AttributeError: 'CLIPModel' object has no attribute 'visual'

In [26]:
model_v = model.visual
model_v(random_image).shape

torch.Size([1, 768])

In [27]:
model_v

VisionTransformer(
  (conv1): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
  (patch_dropout): Identity()
  (ln_pre): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  (transformer): Transformer(
    (resblocks): ModuleList(
      (0-23): 24 x ResidualAttentionBlock(
        (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=1024, out_features=1024, bias=True)
        )
        (ls_1): Identity()
        (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (mlp): Sequential(
          (c_fc): Linear(in_features=1024, out_features=4096, bias=True)
          (gelu): QuickGELU()
          (c_proj): Linear(in_features=4096, out_features=1024, bias=True)
        )
        (ls_2): Identity()
      )
    )
  )
  (ln_post): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
)

In [54]:
# Clip
model, tokenizer = load_backbone_and_tokenizer('clip-14')
model = model.to(device)
prompts = 'test'
tokenized_prompts = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(device)
text_features = model.get_text_features(**tokenized_prompts).to(device)
print(text_features.shape)
text_features = model.text_projection(text_features).to(device)
print(text_features.shape)

torch.Size([1, 768])
torch.Size([1, 768])


In [55]:
model

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 768)
      (position_embedding): Embedding(77, 768)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e-05,

In [47]:
model = load_backbone('clip-14').to(device)

In [45]:
print(model)

CLIPVisionTransformer(
  (embeddings): CLIPVisionEmbeddings(
    (patch_embedding): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
    (position_embedding): Embedding(257, 1024)
  )
  (pre_layrnorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  (encoder): CLIPEncoder(
    (layers): ModuleList(
      (0-23): 24 x CLIPEncoderLayer(
        (self_attn): CLIPAttention(
          (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        )
        (layer_norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (mlp): CLIPMLP(
          (activation_fn): QuickGELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias

In [50]:
output = model(random_image)

In [51]:
print(output)

BaseModelOutputWithPooling(last_hidden_state=tensor([[[ 0.0786,  0.3919,  0.3160,  ..., -0.5392, -0.4338,  0.0148],
         [ 0.4899,  0.1720,  0.2148,  ...,  0.2461,  0.0730,  0.5550],
         [ 0.1115,  0.2680,  0.0043,  ...,  0.0711, -0.0301,  0.8122],
         ...,
         [ 0.4768,  0.3447,  0.6328,  ...,  0.3555,  0.0909,  0.0615],
         [-0.2492,  0.1271,  0.3627,  ..., -0.0731, -0.0374,  0.4203],
         [ 0.4061,  0.2815,  0.8260,  ..., -0.2686,  0.4746,  0.6959]]],
       device='cuda:0'), pooler_output=tensor([[ 0.1918,  1.1120,  0.6592,  ..., -0.7345, -0.9643,  0.0746]],
       device='cuda:0'), hidden_states=None, attentions=None)


In [58]:
print(output.last_hidden_state.shape)
print(output.pooler_output.shape)
model.visual_projection(output.pooler_output).shape

torch.Size([1, 257, 1024])
torch.Size([1, 1024])


torch.Size([1, 768])

In [ ]:
# Tokenize and extract text features
if any(b in args.backbone_type for b in ('openclip', 'remoteclip', 'georsclip')):
    tokenized_prompts = tokenizer(prompts).to(device)
    text_features = model.encode_text(tokenized_prompts).to(device)
else:
    tokenized_prompts = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(device)
    text_features = model.get_text_features(**tokenized_prompts).to(device)

# Pass through projection layer to match image embedding dimension
text_features = model.text_projection(text_features)

# Normalize
norm_text_features = F.normalize(text_features, p=2, dim=-1)

In [16]:
print(model)

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 768)
      (position_embedding): Embedding(77, 768)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e-05,

In [3]:
model = load_backbone('clip-14-gpte-1024-epoch50').to(device)

Using checkpoint weights/vlm4rs/gpt_ensemble_1024_e50.pth


In [4]:
model

CLIPVisionTransformer(
  (embeddings): CLIPVisionEmbeddings(
    (patch_embedding): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
    (position_embedding): Embedding(257, 1024)
  )
  (pre_layrnorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  (encoder): CLIPEncoder(
    (layers): ModuleList(
      (0-23): 24 x CLIPEncoderLayer(
        (self_attn): CLIPAttention(
          (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        )
        (layer_norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (mlp): CLIPMLP(
          (activation_fn): QuickGELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias

In [6]:
model.encode_text('test')

AttributeError: 'CLIPVisionTransformer' object has no attribute 'encode_text'

In [13]:
model, tokenizer = load_backbone_and_tokenizer('clip-14-gpte-1024-epoch50')

Using checkpoint weights/vlm4rs/gpt_ensemble_1024_e50.pth


In [14]:
model

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 768)
      (position_embedding): Embedding(77, 768)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e-05,

In [19]:
prompts = 'test'
tokenized_prompts = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True)
text_features = model.get_text_features(**tokenized_prompts)
print(text_features.shape)

torch.Size([1, 768])


In [26]:
random_image = torch.rand(1, 3, 224, 224)

In [28]:
image_features = model.get_image_features(random_image)
image_features.shape

torch.Size([1, 768])

In [29]:
model_v = model.vision_model

In [30]:
model_v.get_image_features(random_image)

AttributeError: 'CLIPVisionTransformer' object has no attribute 'get_image_features'

### Custom clip

In [35]:
import torch
from models.custom_clip.wrapper import CustomCLIPWrapper
from transformers import CLIPModel, CLIPTokenizer

In [30]:
ckpt_path = '/home/gridsan/manderson/ovdsat/weights/vlm4rs/customclip.ckpt'

In [31]:
checkpoint = torch.load(ckpt_path, map_location=torch.device('cpu'))

/state/partition1/slurm_tmp/28007359.0.0/ipykernel_2311764/2971712510.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(ckpt_path, map_location=tor

In [32]:
print(checkpoint.keys())

dict_keys(['epoch', 'global_step', 'pytorch-lightning_version', 'state_dict', 'callbacks', 'optimizer_states', 'lr_schedulers', 'native_amp_scaling_state'])


In [33]:
print(checkpoint['state_dict'].keys())

odict_keys(['sink_temp', 'model.positional_embedding', 'model.text_projection', 'model.logit_scale', 'model.token_embedding.weight', 'model.ln_final.weight', 'model.ln_final.bias', 'model.visual_projection.weight', 'model.visual.embeddings.class_embedding', 'model.visual.embeddings.patch_embedding.weight', 'model.visual.embeddings.position_embedding.weight', 'model.visual.pre_layrnorm.weight', 'model.visual.pre_layrnorm.bias', 'model.visual.encoder.layers.0.self_attn.k_proj.weight', 'model.visual.encoder.layers.0.self_attn.k_proj.bias', 'model.visual.encoder.layers.0.self_attn.v_proj.weight', 'model.visual.encoder.layers.0.self_attn.v_proj.bias', 'model.visual.encoder.layers.0.self_attn.q_proj.weight', 'model.visual.encoder.layers.0.self_attn.q_proj.bias', 'model.visual.encoder.layers.0.self_attn.out_proj.weight', 'model.visual.encoder.layers.0.self_attn.out_proj.bias', 'model.visual.encoder.layers.0.layer_norm1.weight', 'model.visual.encoder.layers.0.layer_norm1.bias', 'model.visual.e

In [37]:
model = CLIPModel.from_pretrained("/home/gridsan/manderson/ovdsat/weights/clip-vit-large-patch14")
img_encoder = model.vision_model
txt_encoder = model.text_model
    
tokenizer = CLIPTokenizer.from_pretrained("/home/gridsan/manderson/ovdsat/weights/clip-vit-large-patch14")

In [38]:
model = CustomCLIPWrapper(
        img_encoder,
        txt_encoder,
        minibatch_size=1,
        model_name='ViT-L/14',
        avg_word_embs=True, #change to false
    )

FileNotFoundError: [Errno 2] No such file or directory: 'models/configs/ViT.yaml'

In [3]:
import torch

In [4]:
ckpt_path = '/home/gridsan/manderson/train-CLIP/run/fmow/last.ckpt'

In [6]:
ckpt = torch.load(ckpt_path, map_location=torch.device('cpu'))

/state/partition1/slurm_tmp/28007359.0.0/ipykernel_2311764/2654216889.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location=torch.dev

In [7]:
ckpt.keys()

dict_keys(['epoch', 'global_step', 'pytorch-lightning_version', 'state_dict', 'callbacks', 'optimizer_states', 'lr_schedulers', 'native_amp_scaling_state'])

In [24]:
ckpt['state_dict'].keys()

odict_keys(['model.logit_scale', 'model.text_model.embeddings.token_embedding.weight', 'model.text_model.embeddings.position_embedding.weight', 'model.text_model.encoder.layers.0.self_attn.k_proj.weight', 'model.text_model.encoder.layers.0.self_attn.k_proj.bias', 'model.text_model.encoder.layers.0.self_attn.v_proj.weight', 'model.text_model.encoder.layers.0.self_attn.v_proj.bias', 'model.text_model.encoder.layers.0.self_attn.q_proj.weight', 'model.text_model.encoder.layers.0.self_attn.q_proj.bias', 'model.text_model.encoder.layers.0.self_attn.out_proj.weight', 'model.text_model.encoder.layers.0.self_attn.out_proj.bias', 'model.text_model.encoder.layers.0.layer_norm1.weight', 'model.text_model.encoder.layers.0.layer_norm1.bias', 'model.text_model.encoder.layers.0.mlp.fc1.weight', 'model.text_model.encoder.layers.0.mlp.fc1.bias', 'model.text_model.encoder.layers.0.mlp.fc2.weight', 'model.text_model.encoder.layers.0.mlp.fc2.bias', 'model.text_model.encoder.layers.0.layer_norm2.weight', 'm

In [9]:
from transformers import CLIPModel

/home/gridsan/manderson/.conda/envs/train-clip/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
clipmodel = CLIPModel.from_pretrained('/home/gridsan/manderson/ovdsat/weights/clip-vit-large-patch14')

In [26]:
state_dict = ckpt['state_dict']
new_state_dict = {k[len("model."):] if k.startswith("model.") else k: v for k, v in state_dict.items()}
new_state_dict.keys()

dict_keys(['logit_scale', 'text_model.embeddings.token_embedding.weight', 'text_model.embeddings.position_embedding.weight', 'text_model.encoder.layers.0.self_attn.k_proj.weight', 'text_model.encoder.layers.0.self_attn.k_proj.bias', 'text_model.encoder.layers.0.self_attn.v_proj.weight', 'text_model.encoder.layers.0.self_attn.v_proj.bias', 'text_model.encoder.layers.0.self_attn.q_proj.weight', 'text_model.encoder.layers.0.self_attn.q_proj.bias', 'text_model.encoder.layers.0.self_attn.out_proj.weight', 'text_model.encoder.layers.0.self_attn.out_proj.bias', 'text_model.encoder.layers.0.layer_norm1.weight', 'text_model.encoder.layers.0.layer_norm1.bias', 'text_model.encoder.layers.0.mlp.fc1.weight', 'text_model.encoder.layers.0.mlp.fc1.bias', 'text_model.encoder.layers.0.mlp.fc2.weight', 'text_model.encoder.layers.0.mlp.fc2.bias', 'text_model.encoder.layers.0.layer_norm2.weight', 'text_model.encoder.layers.0.layer_norm2.bias', 'text_model.encoder.layers.1.self_attn.k_proj.weight', 'text_mo

In [27]:
# Load state_dict into the CLIP model
clipmodel.load_state_dict(new_state_dict, strict=False)  # strict=False ignores non-matching keys

# Set model to evaluation mode
#clipmodel.eval()

<All keys matched successfully>

In [28]:
torch.save(new_state_dict, '/home/gridsan/manderson/ovdsat/weights/vlm4rs/test.pth')

### CoOp

In [19]:
checkpoint_path = "/home/gridsan/manderson/ovdsat/CoOp/output/dior/CoOp/vit_l14_fmow_5shots/nctx16_cscTrue_ctpend/seed1/prompt_learner/model.pth.tar-50"
checkpoint = torch.load(checkpoint_path, map_location="cpu")

# Extract the learned prompts
learned_prompts = checkpoint["state_dict"]["ctx"]
print(learned_prompts.shape)

torch.Size([20, 16, 768])


/state/partition1/slurm_tmp/47746.0.0/ipykernel_497790/1348725985.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location="